# ⚔️ B3　Boss 戰：班級成績分析
**Python 冒險之旅 2026**　｜　Day 3（08/31 一）🌋 迴圈之島　｜　Boss 戰　｜　🏅 200 XP

📖 對應教科書：第 5–6 章綜合


### 🎯 這一關你會學到
- 用迴圈與串列完成統計、搜尋與排序

### 🧭 闖關方式
1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/python-quest-2026/)。

> 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  Python 冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins

_LEVEL = "B3"
_SALT = "python-quest-2026-datama"
_TASKS = ["B3-1", "B3-2", "B3-3", "B3-4", "B3-5"]
_XP_EACH = 40
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_pyquest_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

def 行列表(out):
    return [ln.rstrip() for ln in str(out).splitlines() if ln.strip()]

class _NeedMoreInput(Exception):
    pass

_HIST = builtins.__dict__.setdefault("_pyquest_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_pyquest_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_pyquest_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

def _find_cell(tid):
    marker = "# 🎯 任務 " + tid
    for cell in reversed(_history()):
        if marker in cell:
            lines = [ln for ln in cell.splitlines()
                     if not re.match(r"\s*(檢查|通關密語)\s*\(", ln)]
            return "\n".join(lines)
    return None

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                _plt.show = _orig_show
        return buf.getvalue(), ns
    run.src = src
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _progress():
    done = sum(1 for t in _TASKS if _PASSED.get(t))
    bar = "■" * done + "□" * (len(_TASKS) - done)
    return f"[{bar}] {done}/{len(_TASKS)}"

def 檢查(tid):
    tid = str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    src = _find_cell(tid)
    if src is None:
        print(f"❌ 找不到「# 🎯 任務 {tid}」的程式格。請先執行那一格（並保留第一行的標記），再執行這裡。")
        return
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        result = (False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。")
    except Exception as e:
        tb = traceback.format_exc().strip().splitlines()[-1]
        result = (False, f"程式執行時發生錯誤 → {tb}")
    ok, extra = (result, "") if isinstance(result, bool) else result
    if ok:
        first = not _PASSED.get(tid)
        _PASSED[tid] = True
        print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")
        if all(_PASSED.get(t) for t in _TASKS):
            print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
    else:
        print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
        if extra: print("   💬 " + str(extra))
        if _HINTS.get(tid): print("   💡 提示：" + _HINTS[tid])
        print("   👉 修改程式後，先重新執行任務那一格，再執行這一格。")

def 通關密語():
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_SALT}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：PYQ-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_B3_1(run):
    out, ns = run()
    if abs(ns.get("avg", 0) - 73.25) > 0.01: return (False, "avg 應該是 73.25。")
    if (ns.get("top_name"), ns.get("low_name")) != ("靜香", "胖虎"): return (False, "最高是靜香、最低是胖虎。")
    return 出現(out, "平均73.2分", "靜香95", "胖虎47")
任務定義("B3-1", _check_B3_1, 提示="用索引 i 比較 scores[i] 與 scores[top_i]。")

def _check_B3_2(run):
    out, ns = run()
    if ns.get("passed") != 6: return (False, "及格人數應該是 6。")
    return (ns.get("failed") == ["阿華", "胖虎"] and 出現(out, "阿華、胖虎"), "不及格名單應該是 阿華、胖虎。")
任務定義("B3-2", _check_B3_2, 提示="條件 scores[i] >= 60。")

def _check_B3_3(run):
    out, ns = run()
    lines = 行列表(out)
    want = ["第1名 靜香 95", "第2名 小美 92", "第3名 小芳 88", "第4名 小明 78", "第5名 小夫 70", "第6名 大雄 61", "第7名 阿華 55", "第8名 胖虎 47"]
    got = [ln for ln in lines if ln.startswith("第")]
    return (got == want, "排名順序或格式不對，應該由高到低。")
任務定義("B3-3", _check_B3_3, 提示="pairs.sort(reverse=True) 會依分數由大到小排。")

def _check_B3_4(run):
    out, ns = run()
    return (ns.get("count") == [2, 1, 2, 1, 2] and 出現(out, "A:2B:1C:2D:1E:2"), f"count 應該是 [2, 1, 2, 1, 2]，現在是 {ns.get('count')}。")
任務定義("B3-4", _check_B3_4, 提示="用 elif 由高到低判斷，對應的 count 索引 +1。")

def _check_B3_5(run):
    out, ns = run()
    lines = 行列表(out)
    if len(lines) != 8: return (False, "應該有 8 行。")
    for ln, n, s in zip(lines, ['小明', '小美', '阿華', '小芳', '大雄', '靜香', '胖虎', '小夫'], [78, 92, 55, 88, 61, 95, 47, 70]):
        if ln.count('█') != s // 10 or not ln.startswith(n) or not ln.endswith(str(s)):
            return (False, f"{n} 那一行應該有 {s // 10} 個 █，最後是分數 {s}。")
    return True
任務定義("B3-5", _check_B3_5, 提示="bar = '█' * (scores[i] // 10)。")


## ⚔️ Boss 登場：班級成績分析
你接下「班級成績分析師」的委託。資料如下（請先執行下面的資料格）：

In [ ]:
names  = ['小明', '小美', '阿華', '小芳', '大雄', '靜香', '胖虎', '小夫']
scores = [78, 92, 55, 88, 61, 95, 47, 70]
print(len(names), '位學生，資料準備完成')

### 🎯 任務 B3-1　基本統計

用迴圈（或內建函式）算出 `avg`（平均，小數 1 位顯示）、`top_name`（最高分的人）、`low_name`（最低分的人），印出：
```
平均 73.2 分
最高分：靜香 95 分
最低分：胖虎 47 分
```

In [ ]:
# 🎯 任務 B3-1　基本統計（請保留這一行）
names  = ['小明', '小美', '阿華', '小芳', '大雄', '靜香', '胖虎', '小夫']
scores = [78, 92, 55, 88, 61, 95, 47, 70]
avg = ???
top_i = 0
low_i = 0
for i in range(len(scores)):
    # 更新 top_i 與 low_i
top_name = names[top_i]
low_name = names[low_i]
print(f"平均 {avg:.1f} 分")
print(f"最高分：{top_name} {scores[top_i]} 分")
print(f"最低分：{low_name} {scores[low_i]} 分")

In [ ]:
檢查("B3-1")   # ◀ 執行這一格，看看任務 B3-1 有沒有過關

### 🎯 任務 B3-2　及格名單

印出及格（≥ 60）人數與**不及格名單**：
```
及格人數：6 / 8
不及格名單：阿華、胖虎
```

In [ ]:
# 🎯 任務 B3-2　及格名單（請保留這一行）
names  = ['小明', '小美', '阿華', '小芳', '大雄', '靜香', '胖虎', '小夫']
scores = [78, 92, 55, 88, 61, 95, 47, 70]
passed = 0
failed = []
for i in range(len(scores)):
    if ???:
        passed += 1
    else:
        failed.append(names[i])
print(f"及格人數：{passed} / {len(scores)}")
print("不及格名單：" + "、".join(failed))

In [ ]:
檢查("B3-2")   # ◀ 執行這一格，看看任務 B3-2 有沒有過關

### 🎯 任務 B3-3　成績排名

印出由高到低的排名表（名次、姓名、分數）。提示：先把 `(分數, 姓名)` 配對成串列，再用 `sort(reverse=True)`。
```
第1名 靜香 95
第2名 小美 92
...
```

In [ ]:
# 🎯 任務 B3-3　成績排名（請保留這一行）
names  = ['小明', '小美', '阿華', '小芳', '大雄', '靜香', '胖虎', '小夫']
scores = [78, 92, 55, 88, 61, 95, 47, 70]
pairs = []
for i in range(len(names)):
    pairs.append([scores[i], names[i]])
pairs.sort(???)
for rank, p in enumerate(pairs, start=1):
    print(f"第{rank}名 {p[1]} {p[0]}")

In [ ]:
檢查("B3-3")   # ◀ 執行這一格，看看任務 B3-3 有沒有過關

### 🎯 任務 B3-4　成績分布

統計各等級人數：A（≥ 90）、B（80～89）、C（70～79）、D（60～69）、E（< 60），印成 `A:2 B:1 C:2 D:1 E:2`。

In [ ]:
# 🎯 任務 B3-4　成績分布（請保留這一行）
scores = [78, 92, 55, 88, 61, 95, 47, 70]
count = [0, 0, 0, 0, 0]     # A, B, C, D, E
for s in scores:
    if s >= 90:
        count[0] += 1
    # 補上其餘等級
print(f"A:{count[0]} B:{count[1]} C:{count[2]} D:{count[3]} E:{count[4]}")

In [ ]:
檢查("B3-4")   # ◀ 執行這一格，看看任務 B3-4 有沒有過關

### 🎯 任務 B3-5　文字長條圖

把每位學生的分數畫成文字長條圖：每 10 分一個 `█`（用整數除法），姓名靠左佔 4 格：
```
小明   ███████ 78
小美   █████████ 92
...
```

In [ ]:
# 🎯 任務 B3-5　文字長條圖（請保留這一行）
names  = ['小明', '小美', '阿華', '小芳', '大雄', '靜香', '胖虎', '小夫']
scores = [78, 92, 55, 88, 61, 95, 47, 70]
for i in range(len(names)):
    bar = ???
    print(f"{names[i]:<4} {bar} {scores[i]}")

In [ ]:
檢查("B3-5")   # ◀ 執行這一格，看看任務 B3-5 有沒有過關

## 🌟 週中任務（9/1–9/2 自學挑戰，Day 4 分享）
1. **加權成績**：新增 `weights = [0.3, 0.3, 0.4]` 與三科成績的二維串列，算出每人加權總分並排名。
2. **搜尋功能**：用 `while True` 做一個可以重複查詢姓名成績的小程式，輸入 `q` 結束。
3. **生活應用**：挑一個你生活中「重複做的事」（記帳、排班、抽籤……），用迴圈和串列寫成小工具，Day 4 上台 1 分鐘分享。

---
## 🔑 通關密語

全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：🧩 L08 函式與模組** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/python-quest-2026/blob/main/notebooks/L08_functions_modules.ipynb)

回到入口網頁：https://johnnychao.github.io/python-quest-2026/